# Apache Spark with PySpark (Google Colab)

In Sessions 1 and 2 you ran **MapReduce** jobs on Hadoop. Today you meet its successor: **Apache Spark** keeps data **in memory** between steps and lets you express a whole
pipeline in a few lines. You will use it three ways: with **RDDs** (the low-level API), with **DataFrames / SQL** (the high-level API) and... to have some fun with maths and literature. 🎲 📖

| Part | What you will do |
|---|---|
| 0 | 🔁 Install Java + PySpark and start a `SparkSession` (ready-made cells) |
| 1 | **RDDs**: transformations, actions, lazy evaluation, partitions — and a 🎯 *Monte Carlo* estimate of **π** you can *see* |
| 2 | 📖 **Word count** on *Don Quijote*: the MapReduce classic in 4 lines, a **word cloud** and the duel *Quijote vs Sancho* |
| 3 | **DataFrames & SQL** on 550 000 *Black Friday* purchases |
| 4 | 🧩 **Final exercise**: your own analysis of the Black Friday data |

**How it works.** Same as before: every step has a *goal* and the *expected result*, the code cells are **empty: you write the code**
(except Part 0 and the cells marked 🎨, which draw the charts and are ready-made), and every question has a box below it where **you write the answer**. Run the cells **in order**.

> ⚠️ A Colab runtime is temporary: if it is recycled you lose the variables and the files. Run Part 0 again and continue from where you were.

# Part 0 – Install Spark (ready-made)

Spark runs on the JVM, so we need Java 17; PySpark (the Python API) is installed with `pip` and **already contains Spark 4.0.1**, so no manual download is needed.
Run the three cells below, in order (≈ 2 min). The last one creates the `SparkSession`: the *entry point* of every Spark program (`spark`), and its `SparkContext` (`sc`), used by the RDD API.
We run Spark in **local mode** with `local[*]`: one JVM that uses all the CPU cores of the Colab machine as if they were workers.

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
pip uninstall -y -q dataproc-spark-connect 2>/dev/null || true
pip install -q pyspark==4.0.1 wordcloud
java -version 2>&1 | head -n 1

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Session3")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
print("Spark", spark.version, "| cores (default parallelism):", sc.defaultParallelism)

# Part 1 – RDDs: the heart of Spark

An **RDD** (*Resilient Distributed Dataset*) is a collection split into **partitions** that live on different workers. You work on it with two kinds of operations:
**transformations** (`map`, `filter`, `flatMap`, `reduceByKey`...) build a *new* RDD and are **lazy** (nothing is computed yet), and **actions** (`count`, `reduce`, `collect`, `take`...) trigger the real computation.

## 1.1 Your first RDD

Create an RDD with the numbers **1 to 1 000 000** (`sc.parallelize`) and answer two things: how many **partitions** it has, and the **sum of the squares of the even numbers** (use `filter`, `map` and `reduce`).

<details><summary>💡 Hint</summary>

`sc.parallelize(range(1, 1000001))`, `rdd.getNumPartitions()`. The pipeline is `rdd.filter(...).map(...).reduce(lambda a, b: a + b)`.

</details>

✅ **Expected:** Partitions = number of cores of the machine (2 on a standard Colab) and the sum is `166667166667000000`. `filter` and `map` are transformations (lazy); `reduce` is the action that made Spark run the job.

In [ ]:
# Your code here

❓ **Question:** Which of the operations you used are transformations and which one is the action? Where did the 'work' happen?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 1.2 Laziness and caching

Build an RDD that is **expensive to compute** (for example, `sc.parallelize(range(3_000_000))` with a `map` that calls a small function doing some maths) and time (`time.time()`) **three** things:
(a) creating the transformation, (b) the first `count()`, (c) a second `count()`. Then `.cache()` the RDD, run `count()` twice more and compare the times.

<details><summary>💡 Hint</summary>

Take `t0 = time.time()` before and print `time.time() - t0` after each step. `cache()` is itself lazy: the first action after it fills the cache.

</details>

✅ **Expected:** (a) ≈ 0 s; (b), (c) and (d) of the same order (a few seconds: without cache Spark *recomputes* the whole pipeline on every action; (b) also pays the start-up of the Python workers); (e) clearly the fastest (≈ 1 s: the data is already in memory).

In [ ]:
# Your code here

❓ **Question:** Why is (a) instantaneous? Why is the second `count()` after `cache()` much faster than the first one?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 🎯 1.3 Showcase: estimate **π** by throwing darts

Throw random points in the square `[0,1] × [0,1]`. The fraction that falls **inside the quarter circle** `x² + y² < 1` is the ratio of the areas: **π/4**. So `π ≈ 4 × inside / total`.
Perfect for Spark: every point is independent, so the work can be split among all the partitions (it is *embarrassingly parallel*).

Write the Spark job (name the function `inside`, the next cell uses it): generate **N = 2 000 000** points with `sc.parallelize(range(N))`, keep with `filter` those that fall inside the circle and count them.

In [ ]:
# Your code here

✅ **Expected:** something like `π ≈ 3.14...` (the third decimal changes on every run: it is random).

<details><summary>💡 Hint</summary>

Define a function `inside(_)` that draws `x, y = random.random(), random.random()` and returns `x*x + y*y < 1`; then `sc.parallelize(range(N)).filter(inside).count()`.

</details>

### 🎨 Now *see* it

Ready-made cell (just run it). Left: 5 000 darts, **green** inside the circle and **red** outside. Right: how the estimate **converges** to π when you throw more darts (it uses Spark for every N).

In [ ]:
import numpy as np, matplotlib.pyplot as plt

rng = np.random.default_rng(7)
x, y = rng.random(5000), rng.random(5000)
ins = x * x + y * y < 1

Ns = [10**k for k in range(2, 7)]              # 100 ... 1 000 000 darts
est = [4 * sc.parallelize(range(n)).filter(inside).count() / n for n in Ns]

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 5.2))
a.scatter(x[ins], y[ins], s=7, c="#2ca02c", label="inside")
a.scatter(x[~ins], y[~ins], s=7, c="#d62728", label="outside")
t = np.linspace(0, np.pi / 2, 200); a.plot(np.cos(t), np.sin(t), "k", lw=2)
a.set_aspect("equal"); a.set_title("5 000 darts: %.0f%% inside -> π ≈ %.3f" % (100 * ins.mean(), 4 * ins.mean()))
a.legend(loc="lower left")
b.semilogx(Ns, est, "o-", lw=2, color="#1f77b4", label="Spark estimate")
b.axhline(np.pi, color="k", ls="--", label="π")
b.set_xlabel("number of darts N"); b.set_ylabel("estimate of π"); b.set_title("More darts, better estimate"); b.legend()
plt.tight_layout(); plt.show()
print({n: round(float(e), 4) for n, e in zip(Ns, est)})

❓ **Question:** Why does the estimate fluctuate a lot with N = 100 and hardly at all with N = 1 000 000? How would you make the job run on more workers without changing the code of `inside`?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

# Part 2 – 📖 Word count on *Don Quijote*

Remember the **WordCount** of MapReduce? Mapper: `(word, 1)`; Reducer: add the ones. In Spark it is **one chain of transformations**: `flatMap` → `map` → `reduceByKey`.
We will use the first chapters of *El ingenioso hidalgo don Quijote de la Mancha* (Cervantes, public domain: about 32 000 words).

## 2.1 Load the book as an RDD

Download the text (`wget`) from `https://github.com/AntSanVar/DistributedSystemData/raw/refs/heads/main/Session03/quijote.txt` and read it as an RDD of **lines** with `sc.textFile`. Print the number of lines, the number of partitions and the **first 3 non-empty lines**.

<details><summary>💡 Hint</summary>

`!wget -q <url>` and `sc.textFile('quijote.txt')`. For the lines: `lines.filter(lambda l: l.strip() != '').take(3)`.

</details>

✅ **Expected:** `lines: 3516 | partitions: 2` and the first non-empty lines: the title *El ingenioso hidalgo don Quijote de la Mancha*, *por Miguel de Cervantes Saavedra*, and the title again (it appears twice in the original file). Each element is **one line** (a `str`); here the file is local, in a cluster it would come from HDFS (`hdfs://...`) split by blocks.

In [ ]:
# Your code here

❓ **Question:** What is each element of the RDD you created? Is the file already 'distributed'?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 2.2 WordCount in four lines

Count how many times each word appears: convert to **lower case**, split each line into words (use the regular expression `[a-záéíóúñü]+` with `re.findall`), make `(word, 1)` pairs and add them with `reduceByKey`. Print the **10 most frequent** words (`takeOrdered(10, key=lambda kv: -kv[1])`).

<details><summary>💡 Hint</summary>

`words = lines.flatMap(lambda l: re.findall(r'[a-záéíóúñü]+', l.lower()))`; `counts = words.map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b)`.

</details>

✅ **Expected:** `total words: 31587 | distinct words: 5348` and a top 10 made only of tiny words: `que` (1676), `de` (1641), `y` (1460), `la`, `el`, `a`, `en`... `map`/`flatMap` ≈ Mapper, `reduceByKey` ≈ Combiner + Reducer. `flatMap` because every line gives *several* words (and `map` would give one *list* per line).

In [ ]:
# Your code here

❓ **Question:** Which transformation plays the role of the Mapper and which one of the Reducer? Why do we use `flatMap` and not `map` to get the words?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 2.3 Throw away the 'stop words'

The top 10 says nothing about the book. Filter out the *stop words* (articles, prepositions, pronouns...) and the words of less than 3 letters, and print the **top 15** again.
The list is ready for you:

```python
STOP = set("""que de y la el en no se los con su las por un una lo al del le es me mi como más pero sus si
        ya este esta esto ese eso aquel aquella para sin sobre entre había ser fue hay muy tan todo
        todos otra otro así cuando donde quien qué nos les ni dijo porque bien era cual pues estaba""".split())
```

<details><summary>💡 Hint</summary>

`clean = counts.filter(lambda kv: kv[0] not in STOP and len(kv[0]) > 2)` and `takeOrdered` again. Keep `clean`: you will draw it next.

</details>

✅ **Expected:** The top is now `don` (256), `quijote` (240), `caballero` (111), `sancho` (97), `vuestra`, `señor`, `respondió`, `merced`, `libro`...: a knight, his squire and a lot of courtesy (*vuestra merced*, *señor*, *don*).

In [ ]:
# Your code here

❓ **Question:** Which words remain at the top? What does this tell you about the book?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

### 🎨 The word cloud

Ready-made cell. The size of every word is proportional to its count: your RDD `clean` is the input (`collect()` brings it to the *driver*, which is fine because it is small).

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

freq = dict(clean.takeOrdered(150, key=lambda kv: -kv[1]))
wc = WordCloud(width=1200, height=600, background_color="#1a1a2e", colormap="Wistia",
               prefer_horizontal=0.9, random_state=3).generate_from_frequencies(freq)
plt.figure(figsize=(12, 5.6)); plt.imshow(wc, interpolation="bilinear"); plt.axis("off")
plt.title("Don Quijote de la Mancha, computed with Spark", color="white", backgroundcolor="#1a1a2e")
plt.tight_layout(); plt.show()

## 2.4 ⚔️ The duel: Quijote vs Sancho

Who is the **main character**? Using the RDD `counts` (not the text again!), get the number of mentions of `quijote`, `sancho`, `dulcinea` and `rocinante` and print them.

<details><summary>💡 Hint</summary>

`counts.filter(lambda kv: kv[0] in CHARS).collect()`, with `CHARS = {'quijote', 'sancho', 'dulcinea', 'rocinante'}`.

</details>

✅ **Expected:** `quijote` (≈ 240) > `sancho` (≈ 97) > `rocinante` (≈ 35) > `dulcinea` (≈ 24). Yes: `counts` is **not** stored (we did not `cache()` it), so every action runs the whole chain again from `textFile`.

In [ ]:
# Your code here

❓ **Question:** Who wins? And what would be the cost of this query if the RDD `counts` had not been computed before: would Spark have to read the book again?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

In [ ]:
import matplotlib.pyplot as plt
names, vals = zip(*sorted(duel, key=lambda kv: kv[1]))
plt.figure(figsize=(8, 3.6))
plt.barh(names, vals, color=["#8c564b", "#e377c2", "#1f77b4", "#ff7f0e"][-len(names):])
for i, v in enumerate(vals): plt.text(v + 2, i, str(v), va="center")
plt.title("Mentions in the book"); plt.xlim(0, max(vals) * 1.15); plt.tight_layout(); plt.show()

# Part 3 – DataFrames and SQL

RDDs are flexible but Spark knows nothing about what is *inside* them. A **DataFrame** is a distributed table with **columns and types**: Spark can then optimise your query (the **Catalyst** optimiser) and you can even write it in **SQL**.
We will use the *Black Friday* dataset: **550 068 purchases** of a retail shop (customer, product, gender, age range, city category, amount spent...).

## 3.1 Load the CSV

Download `https://github.com/AntSanVar/DistributedSystemData/raw/refs/heads/main/Data/train_BF.zip`, unzip it (you get `train.csv`) and read it in a DataFrame `df` with `spark.read.csv`, using the **header** and letting Spark **infer the schema**. Print the schema, the number of rows and the first 5 rows.

<details><summary>💡 Hint</summary>

`!wget -q` and `!unzip -o -q`; `spark.read.csv('train.csv', header=True, inferSchema=True)`; `df.printSchema()`, `df.count()`, `df.show(5)`.

</details>

✅ **Expected:** A schema with 12 columns (`User_ID: integer`, `Product_ID: string`, `Gender`, `Age: string` (a range such as `26-35`), ..., `Purchase: integer`), `rows: 550068` and 5 sample rows. `inferSchema` makes Spark read the file once to guess the types (a *job* runs); the rest is lazy.

In [ ]:
# Your code here

❓ **Question:** What type did Spark infer for `Purchase` and for `Age`? Is reading the CSV a transformation or an action? (Look at what happened when you ran the cell.)

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 3.2 Aggregate with the DataFrame API

How much money did each **city category** (A, B, C) spend, and how many purchases were made there? Use `groupBy` + `agg` with `F.sum` and `F.count`, give each result a name with `alias` and sort by total (descending).

<details><summary>💡 Hint</summary>

`from pyspark.sql import functions as F`; `df.groupBy('City_Category').agg(F.sum('Purchase').alias('total'), F.count('*').alias('purchases')).orderBy(F.desc('total'))`.

</details>

✅ **Expected:** Three rows (B, C, A). B is first both in total and in number of purchases (231 173 purchases; B ≈ 2 115 million, C ≈ 1 664 million, A ≈ 1 316 million).

In [ ]:
# Your code here

❓ **Question:** Which city category spends the most? Is it also the one with the most purchases?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 3.3 The same query in SQL

Register the DataFrame as a **temporary view** (`createOrReplaceTempView`) and write the **same query in SQL** with `spark.sql(...)`.

<details><summary>💡 Hint</summary>

`df.createOrReplaceTempView('purchases')`, then `SELECT City_Category, SUM(Purchase) AS total, COUNT(*) AS purchases FROM purchases GROUP BY City_Category ORDER BY total DESC`.

</details>

✅ **Expected:** The same three rows. The physical plans are (essentially) the same: **both APIs go through Catalyst** and produce the same optimised plan (`HashAggregate`, `Exchange`...).

In [ ]:
# Your code here

❓ **Question:** Is the result identical? Run `.explain()` on both DataFrames (the one of 3.2 and this one): what do you notice about the plans?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

### 🎨 Who spends what?

Ready-made cell: purchases by **age range** and **gender** (the `groupBy` runs in Spark; only the tiny result goes to *pandas* with `toPandas()` to be drawn).

In [ ]:
import matplotlib.pyplot as plt

pdf = (df.groupBy("Age", "Gender").agg(F.sum("Purchase").alias("total"))
         .orderBy("Age").toPandas().pivot(index="Age", columns="Gender", values="total") / 1e6)
ax = pdf.plot(kind="bar", figsize=(10, 4.5), color={"F": "#e377c2", "M": "#1f77b4"}, width=0.8)
ax.set_ylabel("total spent (millions)"); ax.set_title("Black Friday: money spent by age range and gender")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

❓ **Question:** Which age range spends the most? Which gender spends more in total, and does this mean that each *customer* spends more? (hint: how many customers (`User_ID`) of each gender are there? you can count them with `countDistinct`)

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

# Part 4 – 🧩 Final exercise: your own analysis

You are the data analyst of the shop. Answer the question **"which are our 5 star products?"** in three steps.

## 4.1 Top 5 products by revenue (DataFrame API)

Group by `Product_ID` and compute, for each product, the number of purchases (`purchases`), the total revenue (`revenue`) and the average ticket (`avg_ticket`, rounded to 2 decimals). Show the **5 products with the highest revenue**.

<details><summary>💡 Hint</summary>

`F.round(F.avg('Purchase'), 2)`; `orderBy(F.desc('revenue')).limit(5)` or `.show(5)`.

</details>

✅ **Expected:** A table of 5 rows: `P00025442` (1615 purchases, 27 995 166 revenue, average ticket 17 334.47), `P00110742` (1612, 26 722 309), `P00255842` (1383, 25 168 963, avg 18 198.82), `P00059442` (1406, 24 338 343) and `P00184942` (1440, 24 334 887). Note that `P00255842` is third in revenue with fewer purchases than the 4th and 5th: it has the highest average ticket.

In [ ]:
# Your code here

❓ **Question:** Is the product with the highest revenue also the one with the most purchases? Why could they differ?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 4.2 The same in SQL

Write the **same query in SQL** (use the view `purchases` of 3.3) and check that you get the same 5 rows.

<details><summary>💡 Hint</summary>

`ROUND(AVG(Purchase), 2)`, `GROUP BY Product_ID`, `ORDER BY revenue DESC LIMIT 5`.

</details>

✅ **Expected:** The same 5 rows as in 4.1.

In [ ]:
# Your code here

## 4.3 Save the result and count the files

Save the **whole** table `top` (all the products) as **CSV with header** in the folder `top_products` (`write.csv`, mode `overwrite`) after spreading it over **4 partitions** (`repartition(4)`), and list the folder. Then do the same in `top_products_1` with **one partition** (`coalesce(1)`).

<details><summary>💡 Hint</summary>

`top.repartition(4).write.mode('overwrite').csv('top_products', header=True)`; `!ls top_products`. Then `top.coalesce(1).write...`.

</details>

✅ **Expected:** `top_products` has **4** files `part-0000x-....csv` (one per partition, plus `_SUCCESS`); `top_products_1` has just one part file. Each partition is written by one task, in parallel, exactly like each reducer writes its own `part-r-0000x`.

In [ ]:
# Your code here

❓ **Question:** Why does Spark write a *folder* with several `part-...` files and not one single file? How many files do you get in each case and why? (Remember Session 2.)

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

### Your result

Paste the 5 rows of step 4.1 in the box below.

✍️ **Your result** *(double-click, then paste the 5 rows below, one per line inside this quote)*:

> 

# Part 5 – Clean up

## 5.1 Stop Spark

Stop the Spark session to free the memory.

<details><summary>💡 Hint</summary>

`spark.stop()`.

</details>

✅ **Expected:** `Spark stopped`.

In [ ]:
# Your code here

### Final questions

Answer each one in the box below it.

❓ **Question:** What is the difference between a *transformation* and an *action*? Give two examples of each and say what *lazy evaluation* buys you.

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

❓ **Question:** Spark keeps data in memory between steps while MapReduce writes to disk after every job. In which kind of workloads does this make the biggest difference? Name one in which it makes almost none.

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

❓ **Question:** When would you use an RDD and when a DataFrame / SQL?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 